In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import joblib
import re
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import sys

from plotly.subplots import make_subplots

from sklearn.feature_extraction.text import CountVectorizer   # Sac de mots
from sklearn.feature_extraction.text import TfidfVectorizer    # TF-IDF
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS  #Stop words English
)

import nltk
#nltk.download('wordnet', download_dir="./nltk_data")
import shutil
import os
# Nettoyage du dossier wordnet s'il existe (évite les corruptions)
# corpus_dir = './nltk_data/corpora/wordnet'
# if os.path.exists(corpus_dir):
#     shutil.rmtree(corpus_dir)
# Téléchargement des ressources nécessaires pour lemmatizer
nltk.download('wordnet', download_dir="./nltk_data")
nltk.download('omw-1.4', download_dir="./nltk_data")
nltk.download('wordnet_ic', download_dir="./nltk_data")
nltk.data.path.append('./nltk_data')
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer   # Lemmatisation
from nltk.stem.snowball import SnowballStemmer

from contraction_fix import fix as expand_contractions # Contraction





In [ ]:
import spacy                                # NLP avancé
from spacy import displacy                  # Visualisation
from spacy.cli import download
download("en_core_web_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.5 MB/s  0:00:007.5 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
/usr/bin/python


In [5]:
nlp = spacy.load("en_core_web_sm")

In [6]:
df=pd.read_csv('scitweets_export.tsv',
sep='\t')
print (df.head())
print (df.shape)
print (df.columns)

   Unnamed: 0            tweet_id  \
0           0  316669998137483264   
1           1  319090866545385472   
2           2  322030931022065664   
3           3  322694830620807168   
4           4  328524426658328576   

                                                text  science_related  \
0  Knees are a bit sore. i guess that's a sign th...                0   
1          McDonald's breakfast stop then the gym 🏀💪                0   
2  Can any Gynecologist with Cancer Experience ex...                1   
3  Couch-lock highs lead to sleeping in the couch...                1   
4  Does daily routine help prevent problems with ...                1   

   scientific_claim  scientific_reference  scientific_context  
0               0.0                   0.0                 0.0  
1               0.0                   0.0                 0.0  
2               1.0                   0.0                 0.0  
3               1.0                   0.0                 0.0  
4               1.

In [7]:
emoticons_str = r"""
    (?:
        [:=;] # Eyes
        [oO\-]? # Nose (optional)
        [D\)\]\(\]/\\OpP] # Mouth
    )"""

#Prise en compte des éléments qui doivent être regroupés
regex_str = [
    emoticons_str,
    r'<[^>]+>', # HTML tags
    r'(?:@[\w_]+)', # @-mentions
    r"(?:\#+[\w_]+[\w\'_\-]*[\w_]+)", # hash-tags
    r'http[s]?://(?:[a-z]|[0-9]|[$-_@.&amp;+]|[!*\(\),]|(?:%[0-9a-f][0-9a-f]))+', # URLs

    r'(?:(?:\d+,?)+(?:\.?\d+)?)', # nombres
    r"(?:[a-z][a-z'\-_]+[a-z])", # mots avec - et '
    r'(?:[\w_]+)', # autres mots
    r'(?:\S)' # le reste
]

tokens_re = re.compile(r'('+'|'.join(regex_str)+')', re.VERBOSE | re.IGNORECASE)
emoticon_re = re.compile(r'^'+emoticons_str+'$', re.VERBOSE | re.IGNORECASE)

def tokenize(s):
    return tokens_re.findall(s)

def preprocess(s, emoticon=True, lowercase=False):
    tokens = tokenize(s)
    if lowercase:
        tokens = [tok.lower() for tok in tokens]
    if not emoticon:
        tokens = [tok for tok in tokens if not emoticon_re.search(tok)]
    return tokens

In [8]:

print(tokenize(df.text[618]))
print(preprocess(df.text[618], False, True))

['#VIP2', 'Box', 'office', 'Hit', 'With', 'Youths', ',', 'Kids', '&', 'Family', 'Audience', 'Full', 'Support', '#VIP2BlockBuster3RdWeek', '#VIP2BB3rdWeek', 'https://t.co/6VWYClcAtt']
['#vip2', 'box', 'office', 'hit', 'with', 'youths', ',', 'kids', '&', 'family', 'audience', 'full', 'support', '#vip2blockbuster3rdweek', '#vip2bb3rdweek', 'https://t.co/6vwyclcatt']


In [9]:
def stop_words(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []
    for t in tokens:
        key = t.lower()
        if key not in ENGLISH_STOP_WORDS:
            expanded.append(t)
    return " ".join(expanded)

text = "I’m happy but I don't know why."
print(text)
print(stop_words(expand_contractions(text)))

I’m happy but I don't know why.
I'm happy know why.


In [10]:
stemmer = nltk.stem.porter.PorterStemmer()#PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stemmerLemma(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    expanded = []
    t2 = nlp(text)
    
    for t in t2:
        
        key = t.text.lower()
        print(key, t.pos_, stemmer.stem(key), lemmatizer.lemmatize(key))
        if t.pos_ == "VERB":
            result = stemmer.stem(key)
        elif t.pos_ == "NOUN" or t.pos_ == "ADJ" or t.pos_ == "PROPN":
            result = lemmatizer.lemmatize(key)
        else:
            result = key
        expanded.append(result)
    return " ".join(expanded)
print(df.text[3])
print(stemmerLemma(df.text[3]))
text = "I want this cake"
print(stop_words(stemmerLemma(text)))
print(stemmerLemma("I am running"))

Couch-lock highs lead to sleeping in the couch. Gotta stop doing this shit.
couch NOUN couch couch
- PUNCT - -
lock NOUN lock lock
highs NOUN high high
lead VERB lead lead
to ADP to to
sleeping VERB sleep sleeping
in ADP in in
the DET the the
couch NOUN couch couch
. PUNCT . .
got VERB got got
ta PART ta ta
stop VERB stop stop
doing VERB do doing
this DET thi this
shit NOUN shit shit
. PUNCT . .
couch - lock high lead to sleep in the couch . got ta stop do this shit .
i PRON i i
want VERB want want
this DET thi this
cake NOUN cake cake
want cake
i PRON i i
am AUX am am
running VERB run running
i am run
